# Run Ollama on Colab for CanvasUI

Runs `ollama serve` on a free Colab GPU and exposes it publicly via ngrok, so your local CanvasUI dev server can point `OLLAMA_HOST` at it instead of `http://localhost:11434`.

**Before running:** Runtime -> Change runtime type -> select a GPU (T4 is fine).

**You'll need:** a free ngrok account + authtoken from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# 1. Install Ollama (zstd is required by the installer but missing from the base Colab image)
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time

# The official install script also registers ollama as a systemd service that
# auto-starts on port 11434 WITHOUT our OLLAMA_ORIGINS override. Stop/disable it
# first so it can't silently keep answering requests instead of our process.
subprocess.run(["bash", "-c", "systemctl stop ollama 2>/dev/null; systemctl disable ollama 2>/dev/null; true"])
subprocess.run(["bash", "-c", "fuser -k 11434/tcp 2>/dev/null; true"])
time.sleep(3)

env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"

ollama_process = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)
print("ollama serve pid:", ollama_process.pid, "alive:", ollama_process.poll() is None)

ps_output = subprocess.run(["bash", "-c", "ps aux | grep '[o]llama serve'"], capture_output=True, text=True).stdout
print("Processes currently running ollama serve:\n", ps_output)

status = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://localhost:11434/api/tags"],
    capture_output=True, text=True,
).stdout
print("Local check (expect 200):", status)

In [ ]:
# 3. Pull the models used by CanvasUI (src/lib/models.ts)
# Sized for a free Colab T4 (16GB VRAM) -- larger/better than what runs locally on a laptop
!ollama pull qwen2.5vl:7b
!ollama pull qwen2.5-coder:14b

# qwen3.6:27b is ~17GB (Q4_K_M) -- slightly over the free T4's 16GB VRAM, so it will
# partially offload to CPU RAM and run slower than the two models above.
!ollama pull qwen3.6:27b

In [ ]:
# 4. Expose port 11434 publicly via ngrok
!pip install -q pyngrok

from pyngrok import ngrok

NGROK_AUTHTOKEN = ""  # paste your authtoken here
ngrok.set_auth_token(NGROK_AUTHTOKEN)

tunnel = ngrok.connect(11434, "http")
print("Public OLLAMA_HOST:", tunnel.public_url)

## Next steps (on your local machine)

1. Copy the printed `https://....ngrok-free.app` URL.
2. In `.env.local`, set:
   ```
   OLLAMA_HOST=https://your-tunnel-url.ngrok-free.app
   ```
3. Restart `npm run dev`.

**Notes:**
- The tunnel URL changes every time this notebook restarts (free ngrok) -- update `.env.local` again when that happens.
- Keep this notebook tab open/running; closing it or hitting the Colab idle timeout kills the server and tunnel.
- Free Colab sessions disconnect after ~90 min idle / 12 hr max, so this is for testing only, not a permanent setup.